# Pavan et al. (2014) Index-Policy Benchmark — `cont_single`

Estimates the **optimal expected revenue** for the bandit auction (2 buyers, 1 good, T=5, delta=0.9).

**Validity caveat.** PST Proposition 2 assumes (a) non-allocated types are frozen (I=0 when x=0) and (b) I_{i,t,s}=1 in *value space* when allocated. Our logit-space Gaussian satisfies (a) exactly when sigma_no=0, and (b) approximately when types stay near the centre of [0,5]. We report three estimates:

- **Myerson (static)**: per-period Myerson virtual value. *Not* a lower bound — does not satisfy dynamic IC.
- **Pavan Gittins**: Eq. (15) with initial hazard rate. Valid when I=1 in value space.
- **Pavan General**: full Theorem 2 with numerically estimated impulse responses (finite differences at mean type).

In [ ]:
RUN_ENV = 'colab'    # 'colab' | 'local'
LOCAL_PROJECT_DIR = None
# LOCAL_PROJECT_DIR = r'E:\Research\code\Dynamic Mechanism Design\Bandit Auction'

In [ ]:
import os, sys, subprocess

if RUN_ENV == 'colab':
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y',
        'jax', 'jaxlib', 'jax-cuda12-plugin', 'jax-cuda12-pjrt'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-qU',
        'jax[cuda12]', 'dm-haiku', 'optax', 'flax'])
    print('Install done. IMPORTANT: Runtime -> Restart session, then re-run from this cell.')
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        PROJECT_DIR = '/content/drive/MyDrive/My Research/Dynamic Mechanism Design/Bandit Auction'
    except Exception:
        PROJECT_DIR = os.getcwd()
else:
    PROJECT_DIR = LOCAL_PROJECT_DIR or os.getcwd()

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

## Imports & Auction Setup

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from bandit_auction.envs.cont_single import create_cont_single_auction
from bandit_auction.baselines.pavan import (
    eval_myerson_mechanism,
    eval_pavan_bandit_auction,
    eval_pavan_mechanism,
)

SEED = 42
print(f'JAX devices: {jax.devices()}')

In [ ]:
# sigma_no=0.0: non-allocated types exactly frozen — satisfies PST bandit auction assumption.
auction = create_cont_single_auction(
    n_buyers       = 2,
    n_goods        = 1,
    horizon        = 5,
    discount       = 0.9,
    type_mean      = 2.5,
    type_std       = 0.1,
    types_low      = 0.0,
    types_high     = 5.0,
    sigma_no       = 0.0,
    trans_lambda   = 1.0,
    allocation_cost= 0.1,
    seed           = SEED,
)
print(f'Auction: {auction.num_buyers}B x {auction.num_goods}G x T={auction.num_horizons}, '
      f'discount={auction.discount}')
print(f'sigma_no={auction.trans_sigma_no}')

## Myerson (Static) Baseline

Uses the per-period Myerson virtual value psi_i(v)=v-(1-F(v))/f(v) at the current type,
ignoring dynamic impulse-response terms.

**Note:** does *not* satisfy dynamic IC, so it is **not** a lower bound on IC-feasible
dynamic revenue. Reported for reference only.

In [ ]:
NUM_SAMPLES_MYERSON = 2000

print(f'Running Myerson baseline ({NUM_SAMPLES_MYERSON} trajectories)...')
myerson_rev, myerson_surplus = eval_myerson_mechanism(
    auction,
    num_samples = NUM_SAMPLES_MYERSON,
    seed        = SEED,
    mean        = 2.5,
    std         = 0.1,
)
print(f'  Revenue : {myerson_rev:.4f}')
print(f'  Surplus : {myerson_surplus:.4f}')

## Pavan Gittins Benchmark

Implements PST 2014 Eq. (15):
- 1/eta_{i0}(theta_{i0}) computed **once at t=0**, fixed thereafter.
- Gittins stopping-time index with flow payoff (theta_it - c_i - inv_hz_i).
- Revenue = E[sum_t delta^t (theta_it - inv_hz_i - c_i) x_it].

**Valid when** I_{i,t,s}=1 in value space (allocated) and I=0 (non-allocated).
With sigma_no=0 the I=0 condition holds exactly. The I=1 condition holds
approximately when types remain near 2.5; may overestimate if types drift far.

`num_index_samples` controls MC rollouts per Gittins index evaluation.

In [ ]:
NUM_SAMPLES_PAVAN   = 500   # trajectories for revenue estimate
NUM_INDEX_SAMPLES   = 50    # MC rollouts per Gittins index evaluation

print(f'Running Pavan Gittins benchmark ({NUM_SAMPLES_PAVAN} trajectories, '
      f'{NUM_INDEX_SAMPLES} index samples)...')
pavan_rev, pavan_surplus = eval_pavan_bandit_auction(
    auction,
    num_samples       = NUM_SAMPLES_PAVAN,
    num_index_samples = NUM_INDEX_SAMPLES,
    seed              = SEED,
    mean              = 2.5,
    std               = 0.1,
)
print(f'  Revenue : {pavan_rev:.4f}')
print(f'  Surplus : {pavan_surplus:.4f}')

## Pavan General Benchmark (Numerical Impulse Responses)

Uses the full PST 2014 **Theorem 2** formula with impulse responses estimated
numerically via finite differences at the mean type:

  psi_i(v_it, t) = [v_it - (1-F(v_it))/f(v_it)]  (Myerson)
                 - sum_{s>t} delta^{s-t} * I_{t,s} * (1-F(v_it))/f(v_it)

where I_{t,s} = E[dv_is/dv_it | always allocated] is estimated at v=mean.

This corrects for I != 1 in value space due to the logit-space Gaussian transition.
**Limitation:** impulse responses are computed only at the mean type (2.5),
so the correction is approximate when types drift significantly from the mean.

`num_ir_rollouts` controls accuracy of finite-difference IR estimates.

In [ ]:
NUM_SAMPLES_GENERAL = 500
NUM_IR_ROLLOUTS     = 100   # finite-difference rollouts per (t,s) pair

print(f'Running Pavan general benchmark ({NUM_SAMPLES_GENERAL} trajectories, '
      f'{NUM_IR_ROLLOUTS} IR rollouts)...')
pavan_general_rev, pavan_general_surplus = eval_pavan_mechanism(
    auction,
    num_samples     = NUM_SAMPLES_GENERAL,
    num_ir_rollouts = NUM_IR_ROLLOUTS,
    seed            = SEED,
    mean            = 2.5,
    std             = 0.1,
)
print(f'  Revenue : {pavan_general_rev:.4f}')
print(f'  Surplus : {pavan_general_surplus:.4f}')

## Summary

In [ ]:
print('='*62)
print(f'  Benchmark revenue estimates (cont_single, seed={SEED})')
print('='*62)
print(f'  Myerson (static, not dynamic-IC):     {myerson_rev:.4f}')
print(f'  Pavan Gittins  (I=1 assumption)  :    {pavan_rev:.4f}')
print(f'  Pavan General  (numerical IRF)   :    {pavan_general_rev:.4f}')
print()
print('  Suggested PROFIT_GOAL in cont_single_colab.ipynb:')
print(f'    Gittins upper bound -> PROFIT_GOAL = {pavan_rev:.4f}')
print(f'    Corrected estimate  -> PROFIT_GOAL = {pavan_general_rev:.4f}')

## (Optional) Compare with Trained Mechanism

Load a trained mechanism from `cont_single_colab.ipynb` and compare its revenue
against the Pavan benchmark.

In [ ]:
import pickle
from bandit_auction.core import AuctionSimulator, BoundedContPolicy, build_bounded_policy_net
from bandit_auction.mechanisms.network import Mechanism, build_mechanism_network
from bandit_auction.train.evaluate import eval_mechanism_multikey, print_eval_report

EXPERIMENT = 'cont_single'
METHOD     = 'acceptance'   # 'acceptance' | 'penalty'
LOAD_SEED  = 42

load_dir = f'outputs/{EXPERIMENT}/{METHOD}/seed{LOAD_SEED}'

if not os.path.exists(f'{load_dir}/mechanism_params.pkl'):
    print(f'No saved params at {load_dir}/ — run cont_single_colab.ipynb first.')
else:
    with open(f'{load_dir}/mechanism_params.pkl', 'rb') as f:
        mechanism_params = pickle.load(f)
    with open(f'{load_dir}/policy_params.pkl', 'rb') as f:
        policy_params = pickle.load(f)

    mechanism_network = build_mechanism_network()
    policy_network = build_bounded_policy_net(
        d_priv=auction.d_priv, d_pub=auction.d_pub,
        G=auction.num_goods, hidden=128, residual_init_std=1.0,
    )

    eval_keys = jax.random.split(jax.random.PRNGKey(88888), 32)
    results = eval_mechanism_multikey(
        auction           = auction,
        mechanism_network = mechanism_network,
        mechanism_params  = mechanism_params,
        profit_goal       = pavan_rev,   # use Pavan revenue as the goal
        num_samples       = 256,
        keys              = eval_keys,
        policy_network    = policy_network,
        policy_params     = policy_params,
        is_discrete       = False,
    )
    print_eval_report(results, prefix=f'{EXPERIMENT} ({METHOD}): ')
    print(f'  Pavan benchmark revenue : {pavan_rev:.4f}')
    print(f'  Trained mechanism revenue: {results["profit_stats"]["mean"]:.4f}')
    gap = pavan_rev - results['profit_stats']['mean']
    print(f'  Gap to benchmark        : {gap:+.4f}')